# Azure Key Vault Secrets Sync with Terraform and SPN

Terraform creates the resource group, Key Vault, application, service principal, client secret, RBAC assignments, Vault destination and test association.

> The SPN client secret is sensitive and is stored in Terraform state. Do not commit or share the state or plan files.

In [1]:
%env ARM_SUBSCRIPTION_ID=<azure-subscription-id>
%env ARM_TENANT_ID=<azure-tenant-id>
%env TF_VAR_azure_subscription_id=<azure-subscription-id>

env: ARM_SUBSCRIPTION_ID=<azure-subscription-id>
env: ARM_TENANT_ID=<azure-tenant-id>
env: TF_VAR_azure_subscription_id=<azure-subscription-id>


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((p / ".env" for p in (Path.cwd(), *Path.cwd().parents) if (p / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find .env")
load_dotenv(ENV_FILE)

True

## Authenticate and confirm the Azure subscription

In [3]:
! az login --tenant $ARM_TENANT_ID --subscription $ARM_SUBSCRIPTION_ID
! az account show --query '{subscription:name,subscriptionId:id,tenantId:tenantId}' --output table
! az provider register --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --wait
! az provider show --namespace Microsoft.KeyVault --subscription $ARM_SUBSCRIPTION_ID --query '{namespace:namespace,state:registrationState}' --output table

A web browser has been opened at https://login.microsoftonline.com/<azure-tenant-id>/oauth2/v2.0/authorize. Please continue the login in the web browser. If no web browser is available or if the web browser fails to open, use device code flow with `az login --use-device-code`.

Retrieving subscriptions for the selection...
[
  {
    "cloudName": "AzureCloud",
    "homeTenantId": "<azure-tenant-id>",
    "id": "<azure-subscription-id>",
    "isDefault": true,
    "managedByTenants": [],
    "name": "secret-sync-mapfre-test",
    "state": "Enabled",
    "tenantId": "<azure-tenant-id>",
    "user": {
      "name": "jose.merchan@hashicorp.services",
      "type": "user"
    }
  }
]
Subscription             SubscriptionId                        TenantId
-----------------------  ------------------------------------  ------------------------------------
secret-sync-mapfre-test  <azure-subscription-id>  <azure-tenant-id>
Namespace           State
------------------  ----------
Microsoft.KeyVau

## Initialize and validate

In [4]:
! terraform -chdir=terraform-azure-spn init
! terraform -chdir=terraform-azure-spn validate

Initializing the backend...

Initializing provider plugins...
- Reusing previous version of hashicorp/random from the dependency lock file
- Reusing previous version of hashicorp/time from the dependency lock file
- Reusing previous version of hashicorp/vault from the dependency lock file
- Reusing previous version of hashicorp/azuread from the dependency lock file
- Reusing previous version of hashicorp/azurerm from the dependency lock file
- Using previously-installed hashicorp/time v0.14.0
- Using previously-installed hashicorp/vault v5.10.1
- Using previously-installed hashicorp/azuread v3.9.0
- Using previously-installed hashicorp/azurerm v4.81.0
- Using previously-installed hashicorp/random v3.9.0


Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Te

## Review the plan

In [5]:
! terraform -chdir=terraform-azure-spn plan

data.azuread_client_config.current: Reading...
data.azuread_client_config.current: Read complete after 0s [id=<azure-tenant-id>-04b07795-8ddb-461a-bbee-02f9e1bf7b46-0518caac-617d-4fe4-8cee-2809b74be4c2]
data.azurerm_client_config.current: Reading...
data.azurerm_client_config.current: Read complete after 0s [id=Y2xpZW50Q29uZmlncy9jbGllbnRJZD0wNGIwNzc5NS04ZGRiLTQ2MWEtYmJlZS0wMmY5ZTFiZjdiNDY7b2JqZWN0SWQ9MDUxOGNhYWMtNjE3ZC00ZmU0LThjZWUtMjgwOWI3NGJlNGMyO3N1YnNjcmlwdGlvbklkPTBiMzRmN2NhLTI4N2YtNDFiMi1iOGQyLWM4ZjE2MTgzNzVjYTt0ZW5hbnRJZD0yMzdmYmMwNC1jNTJhLTQ1OGItYWY5Ny1lYWY3MTU3YzBjZDQ=]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azuread_application.secrets_sync will be created
  + resource "azuread_application" "secrets_sync" {
      + app_role_ids                = (known after apply)
      + client_id                   = (known after 

## Apply

The configuration waits 60 seconds after creating the application password and RBAC assignment to allow Microsoft Entra propagation.

In [6]:
! terraform -chdir=terraform-azure-spn apply -auto-approve

data.azuread_client_config.current: Reading...
data.azuread_client_config.current: Read complete after 0s [id=<azure-tenant-id>-04b07795-8ddb-461a-bbee-02f9e1bf7b46-0518caac-617d-4fe4-8cee-2809b74be4c2]
data.azurerm_client_config.current: Reading...
data.azurerm_client_config.current: Read complete after 0s [id=Y2xpZW50Q29uZmlncy9jbGllbnRJZD0wNGIwNzc5NS04ZGRiLTQ2MWEtYmJlZS0wMmY5ZTFiZjdiNDY7b2JqZWN0SWQ9MDUxOGNhYWMtNjE3ZC00ZmU0LThjZWUtMjgwOWI3NGJlNGMyO3N1YnNjcmlwdGlvbklkPTBiMzRmN2NhLTI4N2YtNDFiMi1iOGQyLWM4ZjE2MTgzNzVjYTt0ZW5hbnRJZD0yMzdmYmMwNC1jNTJhLTQ1OGItYWY5Ny1lYWY3MTU3YzBjZDQ=]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # azuread_application.secrets_sync will be created
  + resource "azuread_application" "secrets_sync" {
      + app_role_ids                = (known after apply)
      + client_id                   = (known after 

## Verify Vault and Azure Key Vault

In [7]:
! vault read sys/sync/destinations/azure-kv/mapfre-spn-azure-kv
! vault read -format=json sys/sync/destinations/azure-kv/mapfre-spn-azure-kv/associations | jq

Key                   Value
---                   -----
connection_details    map[client_id:3558142c-769e-4dd1-8b24-09559f7c0ff4 client_secret:***** key_vault_uri:https://kv-mapfre-spn-f95039.vault.azure.net/ tenant_id:<azure-tenant-id>]
name                  mapfre-spn-azure-kv
options               map[custom_tags:map[ManagedBy:Terraform Purpose:VaultSecretsSyncSPN] granularity_level:secret-path secret_name_template:vault-mapfre-spn-{{ .SecretBaseName }}]
type                  azure-kv
{
  "request_id": "7ab53843-827d-af6f-40c5-b7cd96ecfdf4",
  "lease_id": "",
  "lease_duration": 0,
  "renewable": false,
  "data": {
    "associated_secrets": {
      "kv_511eef72/verification": {
        "accessor": "kv_511eef72",
        "external_name": "vault-mapfre-spn-verification",
        "last_operation": "Write",
        "mount": "mapfre-spn-kv",
        "secret_name": "verification",
        "sync_status": "SYNCED",
        "updated_at": "2026-07-23T16:44:46.788185684Z"
      }
    },
    "s

In [8]:
%%bash
KEY_VAULT_NAME=$(terraform -chdir=terraform-azure-spn output -raw key_vault_name)
SECRET_NAME=$(terraform -chdir=terraform-azure-spn output -raw external_secret_name)
az keyvault secret show \
  --vault-name "$KEY_VAULT_NAME" \
  --name "$SECRET_NAME" \
  --query '{name:name,enabled:attributes.enabled,updated:attributes.updated}' \
  --output json

{
  "enabled": true,
  "name": "vault-mapfre-spn-verification",
  "updated": "2026-07-23T16:44:46+00:00"
}


# CLEAN UP

In [9]:
! terraform -chdir=terraform-azure-spn destroy -auto-approve

random_string.suffix: Refreshing state... [id=f95039]
vault_activation_flags.secrets_sync: Refreshing state... [id=secrets-sync]
vault_mount.kv: Refreshing state... [id=mapfre-spn-kv]
vault_kv_secret_v2.demo: Refreshing state... [id=mapfre-spn-kv/data/verification]
data.azuread_client_config.current: Reading...
data.azuread_client_config.current: Read complete after 0s [id=<azure-tenant-id>-04b07795-8ddb-461a-bbee-02f9e1bf7b46-0518caac-617d-4fe4-8cee-2809b74be4c2]
azuread_application.secrets_sync: Refreshing state... [id=/applications/65071d06-9d0a-4834-9f36-a9f690ea17e5]
azuread_application_password.secrets_sync: Refreshing state... [id=65071d06-9d0a-4834-9f36-a9f690ea17e5/password/1d5f65d6-892e-4a7a-9420-81125f7873ac]
azuread_service_principal.secrets_sync: Refreshing state... [id=/servicePrincipals/309be943-310f-4daf-81c3-0884d9d8832b]
data.azurerm_client_config.current: Reading...
azurerm_resource_group.secrets_sync: Refreshing state... [id=/subscriptions/<azure-subscription-id>/re